In [1]:
import pandas as pd
from fredapi import Fred

# Connect to FRED API
with open('api_key.txt', 'r') as f:
    api_key = f.read().strip()  # .strip() removes any accidental newline/whitespace

fred = Fred(api_key=api_key)

# Pull raw series data from FRED by daily and monthly 
# Daily
dxy_raw = fred.get_series('DTWEXBGS')   # Nominal Broad USD Index
vix_raw = fred.get_series('VIXCLS')     # CBOE VIX

# Monthly
ff_raw    = fred.get_series('FEDFUNDS') # US Fed Funds Rate
ecb_raw   = fred.get_series('ECBDFR')   # ECB Deposit Facility Rate
trade_raw = fred.get_series('BOPGSTB')  # US Trade Balance

# Convert the raw series data to DataFrames with named columns
def to_df(series, col_name):
    df = series.rename(col_name).to_frame()
    df.index.name = 'DATE'
    df.index = pd.to_datetime(df.index)
    return df

dxy   = to_df(dxy_raw,   'DXY')
vix   = to_df(vix_raw,   'VIX')
ff    = to_df(ff_raw,    'FEDFUNDS')
ecb   = to_df(ecb_raw,   'ECB_RATE')
trade = to_df(trade_raw, 'TRADE_BAL')

# Handle missing values (FRED uses NaN for gaps / "." entries)
for df in [dxy, vix, ff, ecb, trade]:
    df.dropna(inplace=True)

# Resample data from daily → monthly 

# DXY: last observed value of the month (closing level)
# VIX: monthly mean (average anxiety level over the month)
dxy_m = dxy.resample('ME').last()
vix_m = vix.resample('ME').mean()

# Monthly series: resample to ME just to align index format
ff_m    = ff.resample('ME').last()
ecb_m   = ecb.resample('ME').last()
trade_m = trade.resample('ME').last()

# Merge all dataframes on date index
merged = (dxy_m
    .join(vix_m,   how='inner')
    .join(ff_m,    how='inner')
    .join(ecb_m,   how='inner')
    .join(trade_m, how='inner')
)

# Engineer the interest rate differential predictor
merged['INT_DIFF'] = merged['FEDFUNDS'] - merged['ECB_RATE']

# Create 1-month-ahead DXY target column 
merged['DXY_next'] = merged['DXY'].shift(-1)

# Drop any remaining NaNs (last row loses target; early rows missing ECB)
merged.dropna(inplace=True)

# Reset index and standardize date format 
merged.reset_index(inplace=True)
merged['DATE'] = merged['DATE'].dt.strftime('%Y-%m-%d')

# Save merged dataframe as csv for data analysis
merged.to_csv('dxy_dataset.csv', index=False)

print(f"Dataset shape: {merged.shape}")
print(f"Date range:    {merged['DATE'].min()} → {merged['DATE'].max()}")
print(merged.head())

Dataset shape: (241, 8)
Date range:    2006-01-31 → 2026-01-31
         DATE       DXY        VIX  FEDFUNDS  ECB_RATE  TRADE_BAL  INT_DIFF  \
0  2006-01-31   99.4311  12.036000      4.29      1.25   -66967.0      3.04   
1  2006-02-28   99.7695  12.471053      4.49      1.25   -62308.0      3.24   
2  2006-03-31  100.5600  11.693913      4.59      1.50   -62397.0      3.09   
3  2006-04-30   98.1412  11.847368      4.79      1.50   -62603.0      3.29   
4  2006-05-31   97.7705  14.454545      4.94      1.50   -64818.0      3.44   

   DXY_next  
0   99.7695  
1  100.5600  
2   98.1412  
3   97.7705  
4   98.2483  
